In [1]:
%load_ext autoreload
%autoreload 2
%xmode Plain

Exception reporting mode: Plain


In [2]:
import pandas as pd
import plotly
import plotly.express as px 
import dtale
import numpy as np
import random
from bazaraki import utils
from tqdm import tqdm
import swifter
from pathlib import Path
import datacompy
from datetime import datetime, date
from parse import parse
from glob import glob 


/var/folders/gv/96s0_n4j4xl87r8p93wpnf0r0000gp/T/ipykernel_60203/3232598961.py:11: UserWarning:

Python 3.12 and above currently is not supported by Spark and Ray. Please note that some functionality will not work and currently is not supported.

/var/folders/gv/96s0_n4j4xl87r8p93wpnf0r0000gp/T/ipykernel_60203/3232598961.py:11: UserWarning:

SparkPandasCompare currently only supports Numpy < 2.Please note that the SparkPandasCompare functionality will not work and currently is not supported.



In [3]:
pd.set_option('display.max_rows', 50)  # Disable row limit
pd.set_option('display.max_columns', 40)  # Disable column limit
pd.set_option('display.width', 20)  # Disable line width limit
pd.set_option('display.max_colwidth', 100)  # Disable column width limit
# Set the display precision to 2 decimal places  
pd.set_option('display.precision', 2)  
  
# Or set the float format to always show 2 decimal places  
pd.set_option('display.float_format', '{:.2f}'.format)  


In [4]:
pd.options.plotting.backend = "plotly"
plotly.io.renderers.default = "notebook_connected"


In [5]:
df = utils.read_dfs("output/*.parquet")
# df = utils.read_dfs("output/2025-04-*.parquet")

Reading output/2024-12-12 18:34:25 real-estate-to-rent_real-estate-for-sale.parquet
Reading output/2024-12-14 11:44:22 real-estate-to-rent_real-estate-for-sale.parquet
Total: 35559 valid: 34669 read: 34669 new: 745 deleted: 890 undeleted: 0
Reading output/2024-12-15 18:00:14 real-estate-to-rent_real-estate-for-sale.parquet
Total: 35734 valid: 34398 read: 34398 new: 175 deleted: 490 undeleted: 44
Reading output/2024-12-16 23:13:52 real-estate-to-rent_real-estate-for-sale.parquet
Total: 36281 valid: 34510 read: 34510 new: 547 deleted: 648 undeleted: 213
Reading output/2024-12-17 21:31:02 real-estate-to-rent_real-estate-for-sale.parquet
Total: 37085 valid: 34681 read: 34681 new: 804 deleted: 817 undeleted: 184
Reading output/2024-12-18 23:01:10 real-estate-to-rent_real-estate-for-sale.parquet
Total: 37577 valid: 34809 read: 34809 new: 492 deleted: 468 undeleted: 104
Reading output/2024-12-19 22:09:26 real-estate-to-rent_real-estate-for-sale.parquet
Total: 38047 valid: 33723 read: 33723 ne

In [6]:
df.iloc[0].T

url                                                 https://www.bazaraki.com/adv/5415213_4-bedroom-detached-house-to-rent/
title                                                                                     4-bedroom detached house to rent
price                                                                                                              1650.00
original_price                                                                                                         NaN
price_per_sqm                                                                                                          NaN
location                                                                                               Larnaca, Dromolaxia
posted                                                                                                     Yesterday 20:57
reference_number                                                                                                       NaN
views           

In [7]:
df.cat1.unique()

array(['Houses to rent', 'Apartments, flats to rent', 'Houses for sale',
       'Apartments, flats for sale', 'Commercial property', 'Short term',
       'Plots of land', 'Rooms, flatmates', 'Prefabricated houses',
       'Residential buildings', 'Other'], dtype=object)

In [8]:
df1 = df.query("cat1 == 'Apartments, flats to rent' and not delete_date.isna()").copy()
df1.posted_dt = pd.to_datetime(df1.posted_dt)
df1.delete_date = pd.to_datetime(df1.delete_date)
df1

,url,title,price,original_price,price_per_sqm,location,posted,reference_number,views,lat,lng,sold,cat0,cat1,Property area,Pets,Type,Parking,Plot area,Furnishing,...,Postal code,Construction year,Reference number,Condition,Square meter price,Minimum stay,Land type,Plot Type,Parcel number,Planning zone,Registration number,Share,Density,Coverage,Registration block,Area,Pick a point,posted_dt,delete_date,Property Type
ad_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5508233,https://www.bazaraki.com/adv/5508233_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,1150.00,NaN,NaN,"Paphos, Chlorakas",Yesterday 21:02,NaN,385,34.79,32.42,False,Cyprus real estate to rent,"Apartments, flats to rent",100.00,Allowed,Apartment,Uncovered,NaN,Fully Furnished,...,8220.00,2008,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-11 21:02:00,2025-01-11,NaN
5395481,https://www.bazaraki.com/adv/5395481_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,1100.00,NaN,NaN,"Larnaca, Livadia Larnakas",07.12.2024 11:07,NaN,26,34.95,33.64,False,Cyprus real estate to rent,"Apartments, flats to rent",90.00,Not allowed,Apartment,Covered,NaN,Fully Furnished,...,7060.00,2015,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-07 11:07:00,2025-01-20,NaN
5562261,https://www.bazaraki.com/adv/5562261_1-bedroom-apartment-to-rent/,1-bedroom apartment to rent,720.00,NaN,NaN,"Nicosia, Nicosia - Ag. Antonios",07.12.2024 12:34,NaN,45,35.17,33.37,False,Cyprus real estate to rent,"Apartments, flats to rent",60.00,Not allowed,Apartment,Covered,NaN,Unfurnished,...,NaN,2015,rentonline-107,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-07 12:34:00,2025-02-10,NaN
5525766,https://www.bazaraki.com/adv/5525766_1-bedroom-apartment-to-rent/,1-bedroom apartment to rent,1200.00,NaN,NaN,"Limassol, Polemidia Kato",08.12.2024 19:03,NaN,37,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",65.00,Allowed,Apartment,Covered,NaN,Fully Furnished,...,NaN,2011,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-08 19:03:00,2024-12-22,NaN
5534586,https://www.bazaraki.com/adv/5534586_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,1500.00,NaN,NaN,"Limassol, Limassol - Petrou Kai Pavlou",08.12.2024 19:03,NaN,13,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",85.00,Not allowed,Apartment,Uncovered,NaN,Fully Furnished,...,NaN,2012,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-08 19:03:00,2025-01-14,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5737674,https://www.bazaraki.com/adv/5737674_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,800.00,NaN,NaN,"Nicosia, Aglantzia",13:08,NaN,3,35.15,33.40,False,Cyprus real estate to rent,"Apartments, flats to rent",85.00,Not allowed,Apartment,Uncovered,NaN,Fully Furnished,...,2114.00,2010,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-04-05 13:08:00,2026-08-26,NaN
5737696,https://www.bazaraki.com/adv/5737696_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,900.00,NaN,NaN,"Larnaca, Drosia",13:19,NaN,2,34.92,33.62,False,Cyprus real estate to rent,"Apartments, flats to rent",80.00,Not allowed,Apartment,Covered,NaN,Fully Furnished,...,NaN,None,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-04-05 13:19:00,2026-08-26,NaN
5737713,https://www.bazaraki.com/adv/5737713_studio-apartment-to-rent/,1-bedroom apartment to rent,1000.00,NaN,NaN,"Limassol, Limassol - Zakaki",13:47,NaN,4,34.66,33.01,False,Cyprus real estate to rent,"Apartments, flats to rent",50.00,Allowed,Apartment,Uncovered,NaN,Fully Furnished,...,NaN,2024,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-04-05 13:47:00,2026-08-26,NaN


In [9]:
df1["age_days"] = (df1["delete_date"] - df1["posted_dt"]).dt.days
df1

,url,title,price,original_price,price_per_sqm,location,posted,reference_number,views,lat,lng,sold,cat0,cat1,Property area,Pets,Type,Parking,Plot area,Furnishing,...,Construction year,Reference number,Condition,Square meter price,Minimum stay,Land type,Plot Type,Parcel number,Planning zone,Registration number,Share,Density,Coverage,Registration block,Area,Pick a point,posted_dt,delete_date,Property Type,age_days
ad_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5508233,https://www.bazaraki.com/adv/5508233_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,1150.00,NaN,NaN,"Paphos, Chlorakas",Yesterday 21:02,NaN,385,34.79,32.42,False,Cyprus real estate to rent,"Apartments, flats to rent",100.00,Allowed,Apartment,Uncovered,NaN,Fully Furnished,...,2008,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-11 21:02:00,2025-01-11,NaN,30
5395481,https://www.bazaraki.com/adv/5395481_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,1100.00,NaN,NaN,"Larnaca, Livadia Larnakas",07.12.2024 11:07,NaN,26,34.95,33.64,False,Cyprus real estate to rent,"Apartments, flats to rent",90.00,Not allowed,Apartment,Covered,NaN,Fully Furnished,...,2015,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-07 11:07:00,2025-01-20,NaN,43
5562261,https://www.bazaraki.com/adv/5562261_1-bedroom-apartment-to-rent/,1-bedroom apartment to rent,720.00,NaN,NaN,"Nicosia, Nicosia - Ag. Antonios",07.12.2024 12:34,NaN,45,35.17,33.37,False,Cyprus real estate to rent,"Apartments, flats to rent",60.00,Not allowed,Apartment,Covered,NaN,Unfurnished,...,2015,rentonline-107,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-07 12:34:00,2025-02-10,NaN,64
5525766,https://www.bazaraki.com/adv/5525766_1-bedroom-apartment-to-rent/,1-bedroom apartment to rent,1200.00,NaN,NaN,"Limassol, Polemidia Kato",08.12.2024 19:03,NaN,37,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",65.00,Allowed,Apartment,Covered,NaN,Fully Furnished,...,2011,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-08 19:03:00,2024-12-22,NaN,13
5534586,https://www.bazaraki.com/adv/5534586_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,1500.00,NaN,NaN,"Limassol, Limassol - Petrou Kai Pavlou",08.12.2024 19:03,NaN,13,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",85.00,Not allowed,Apartment,Uncovered,NaN,Fully Furnished,...,2012,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-08 19:03:00,2025-01-14,NaN,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5737674,https://www.bazaraki.com/adv/5737674_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,800.00,NaN,NaN,"Nicosia, Aglantzia",13:08,NaN,3,35.15,33.40,False,Cyprus real estate to rent,"Apartments, flats to rent",85.00,Not allowed,Apartment,Uncovered,NaN,Fully Furnished,...,2010,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-04-05 13:08:00,2026-08-26,NaN,507
5737696,https://www.bazaraki.com/adv/5737696_2-bedroom-apartment-to-rent/,2-bedroom apartment to rent,900.00,NaN,NaN,"Larnaca, Drosia",13:19,NaN,2,34.92,33.62,False,Cyprus real estate to rent,"Apartments, flats to rent",80.00,Not allowed,Apartment,Covered,NaN,Fully Furnished,...,None,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-04-05 13:19:00,2026-08-26,NaN,507
5737713,https://www.bazaraki.com/adv/5737713_studio-apartment-to-rent/,1-bedroom apartment to rent,1000.00,NaN,NaN,"Limassol, Limassol - Zakaki",13:47,NaN,4,34.66,33.01,False,Cyprus real estate to rent,"Apartments, flats to rent",50.00,Allowed,Apartment,Uncovered,NaN,Fully Furnished,...,2024,None,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-04-05 13:47:00,2026-08-26,NaN,507


In [10]:
df2 = utils.add_city_disctrict_cols(df1)
df2.iloc[0].T

url               https://www.bazaraki.com/adv/5508233_2-bedroom-apartment-to-rent/
title                                                   2-bedroom apartment to rent
price                                                                       1150.00
original_price                                                                  NaN
price_per_sqm                                                                   NaN
                                                ...                                
delete_date                                                     2025-01-11 00:00:00
Property Type                                                                   NaN
age_days                                                                         30
city                                                                         Paphos
district                                                                  Chlorakas
Name: 5508233, Length: 52, dtype: object

In [11]:
df1.groupby(["city", "Bedrooms"]).agg(
    mean_age_days=("age_days", "mean"),
    median_age_days=("age_days", "median"),
    count=("age_days", "count")
).reset_index().sort_values("median_age_days", ascending=True).round(2).query("count > 10 and Bedrooms in ['1', '2', '3', 'Studio']")

,city,Bedrooms,mean_age_days,median_age_days,count
0,Famagusta,1,92.80,30.50,44
17,Limassol,Studio,131.36,30.50,216
2,Famagusta,3,123.46,32.00,13
23,Nicosia,Studio,141.22,32.00,120
4,Famagusta,Studio,69.77,32.00,13
29,Paphos,Studio,140.15,32.50,66
11,Limassol,1,139.29,33.00,1310
24,Paphos,1,130.48,34.00,315
18,Nicosia,1,149.22,35.00,1330
1,Famagusta,2,137.59,37.50,100


In [12]:
# check examples
df1.query("city == 'Larnaca' and Bedrooms == 'Studio'")

,url,title,price,original_price,price_per_sqm,location,posted,reference_number,views,lat,lng,sold,cat0,cat1,Property area,Pets,Type,Parking,Plot area,Furnishing,...,Condition,Square meter price,Minimum stay,Land type,Plot Type,Parcel number,Planning zone,Registration number,Share,Density,Coverage,Registration block,Area,Pick a point,posted_dt,delete_date,Property Type,age_days,city,district
ad_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3753553,https://www.bazaraki.com/adv/3753553_studio-on-the-sea-in-dekelia/,Studio apartment to rent,580.00,NaN,NaN,"Larnaca, Oroklini",09.12.2024 06:40,NaN,52,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",30.00,Not allowed,Apartment,Uncovered,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-09 06:40:00,2026-08-26,NaN,624,Larnaca,Oroklini
4852027,https://www.bazaraki.com/adv/4852027_studio-apartment-to-rent/,Studio apartment to rent,450.00,500.00,NaN,"Larnaca, Perivolia Larnakas",07.12.2024 13:30,NaN,14537,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",0.00,Not allowed,Apartment,Covered,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-12-07 13:30:00,2025-02-22,NaN,76,Larnaca,Perivolia Larnakas
5416646,https://www.bazaraki.com/adv/5416646_studio-apartment-to-rent/,Studio apartment to rent,600.00,NaN,NaN,"Larnaca, Larnaka - Finikoudes",14.11.2024 00:01,NaN,558,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",25.00,Allowed,Apartment,No,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-11-14 00:01:00,2026-08-26,NaN,649,Larnaca,Larnaka - Finikoudes
5349458,https://www.bazaraki.com/adv/5349458_studio-apartment-to-rent/,Studio apartment to rent,580.00,NaN,NaN,"Larnaca, Larnaka - Makenzy",18.11.2024 14:50,NaN,2313,34.90,33.63,False,Cyprus real estate to rent,"Apartments, flats to rent",24.00,Not allowed,Apartment,Uncovered,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-11-18 14:50:00,2026-08-26,NaN,645,Larnaca,Larnaka - Makenzy
5534674,https://www.bazaraki.com/adv/5534674_studio-apartment-to-rent/,Studio apartment to rent,950.00,NaN,NaN,"Larnaca, Larnaka - Finikoudes",18.11.2024 15:12,NaN,184,34.91,33.63,False,Cyprus real estate to rent,"Apartments, flats to rent",80.00,Not allowed,Apartment,Covered,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,None,2024-11-18 15:12:00,2025-02-04,NaN,77,Larnaca,Larnaka - Finikoudes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5723547,https://www.bazaraki.com/adv/5723547_studio-apartment-to-rent/,Studio apartment to rent,550.00,NaN,NaN,"Larnaca, Larnaka - Harbor",28.03.2025 11:28,NaN,231,34.92,33.64,False,Cyprus real estate to rent,"Apartments, flats to rent",45.00,Not allowed,Apartment,Covered,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-03-28 11:28:00,2026-08-26,NaN,515,Larnaca,Larnaka - Harbor
5723891,https://www.bazaraki.com/adv/5723891_studio-apartment-to-rent/,Studio apartment to rent,1100.00,NaN,NaN,"Larnaca, Larnaka - Makenzy",28.03.2025 14:42,NaN,74,34.90,33.64,False,Cyprus real estate to rent,"Apartments, flats to rent",33.00,None,Apartment,No,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-03-28 14:42:00,2026-08-26,NaN,515,Larnaca,Larnaka - Makenzy
5727562,https://www.bazaraki.com/adv/5727562_studio-apartment-to-rent/,Studio apartment to rent,1000.00,NaN,NaN,"Larnaca, Larnaka - Makenzy",31.03.2025 13:37,NaN,41,NaN,NaN,False,Cyprus real estate to rent,"Apartments, flats to rent",35.00,None,Apartment,Uncovered,NaN,Fully Furnished,...,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,2025-03-31 13:37:00,2026-08-26,NaN,512,Larnaca,Larnaka - 